# Walk-forward cross-match correlations

**The question.** As the tournament goes on, do the prices of different matches move together?

- **Same team, two different matches** (ARG in ARG–DZA, ARG in ARG–AUT): if the market is learning how good ARG is, these should move **together (+)**.
- **A team and its future opponent** (ARG vs AUT, in their *other* matches, when they meet later): good news for ARG is bad news for AUT, so these should move **opposite (−)**.

We compute every pair's correlation in 3-day windows that slide forward one day at a time. Finished matches drop out.

**How to use this notebook.** All the logic lives in `pipeline/learn/hy_walk.py`; this notebook calls it one step at a time so you can look at what each step produces. Read the matching function in the `.py` file alongside each step. Run the cells top to bottom. When you change something, do **Kernel → Restart & Run All** before believing the output.

*Setup, once, in your terminal:* `pip install jupyterlab matplotlib`, then from the data folder run `jupyter lab` and open this file.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The notebook lives in <data>/notebooks/, the module in <data>/pipeline/learn/
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'pipeline', 'learn'))
import hy_walk as hw
print('data folder:', hw.ROOT)

## Step 1 — Load quotes → mid prices
Read `load_quotes()` in the `.py` file first. Each row is the exchange saying "best bid and best ask are now X and Y". The mid is the market's probability × 100.

**Look at:** how many rows the spread filter removes, and the first few rows.

In [ ]:
quotes = hw.load_quotes()
quotes.head()

## Steps 2–3 — Groups and match end times
`fixture_groups()` finds groups as connected clusters of the fixture graph. `match_info()` decides when each match ended — and refuses the fake 21:00 kickoffs.

**Look at:** ~60 matches in 10 groups. Which matches have `trusted_end = False`? (They're the ones cut off by the tape — the June 16 matches and the gap.)

In [ ]:
info = hw.match_info(quotes)
info_df = pd.DataFrame(info).T
info_df['end_utc'] = pd.to_datetime(info_df.end.astype('int64'), unit='ms', utc=True)
info_df[~info_df.trusted_end.astype(bool)]   # .T makes columns "object" dtype; astype(bool) fixes ~

## Steps 4–5 — Series and increments
One series per (match, team): that team's win price, transformed with `2·asin(√p)`, and stored as its **changes** (`Increments`).

**Look at:** one match's price path. Notice how it sits still for long stretches and then jumps — that's why we work with changes and not with a fixed time grid.

In [ ]:
series = hw.build_series(quotes, info, mask_maint=False)

key = ('26JUN11MEXRSA', 'MEX')            # try other keys: list(series)[:10]
q = quotes[(quotes.match == key[0]) & (quotes.leg == key[1])]
fig, ax = plt.subplots(figsize=(10, 3))
ax.step(pd.to_datetime(q.ts, unit='ms', utc=True), q.mid, where='post', lw=0.8)
ko = pd.to_datetime(info[key[0]]['end'], unit='ms', utc=True)
ax.axvline(ko, color='grey', ls='--', lw=0.8, label='final whistle (end)')
ax.set_ylabel('mid (cents)'); ax.set_title(f'{key[1]} to win, {key[0]}'); ax.legend()
plt.show()

inc = series[key]['inc']
print(f'{len(q):,} quotes -> {len(inc.dx):,} price changes')

## Step 6 — Prove the estimator works first
Read `hy_brute` (the obvious double loop) and then `hy` (the fast version). The self-test checks three things:
1. fast == slow,
2. HY recovers a correlation we planted (+0.6, 0, −0.5) while a 1-minute grid Pearson collapses towards 0 — that's the **Epps effect**, the reason we don't use the grid,
3. at true ρ = 0 the average estimate is 0 (no bias).

**Never skip this.** If the estimator is wrong, everything after it is wrong in a way that still looks like numbers.

In [ ]:
hw.selftest()

## Step 7 — Walk forward
This takes 2–3 minutes. If you've already run it once, set `RUN = False` to load the saved result instead.

In [ ]:
RUN = False
NAME = 'walk_forward.parquet'      # change this when you run a variant, so you never overwrite the baseline
path = os.path.join(hw.OUT, NAME)
if RUN or not os.path.exists(path):
    w = hw.walk(series)
    os.makedirs(hw.OUT, exist_ok=True); w.to_parquet(path, index=False)
else:
    w = pd.read_parquet(path)
print(f'{len(w):,} pair-window estimates')
w.head()

## Step 8 — Read the means **with their counts**
A mean of 5 pairs is mostly noise; a mean of 6,000 is not. Always look at both tables together.

In [ ]:
classes = ['same_team', 'future_opponent', 'same_group_other', 'cross_group']
mean = w.pivot_table(index='window_start', columns='cls', values='rho', aggfunc='mean')[classes]
count = w.pivot_table(index='window_start', columns='cls', values='rho', aggfunc='count')[classes]
display(mean.round(3))
display(count.fillna(0).astype(int))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = pd.to_datetime(mean.index)
for c in classes:
    ax.plot(x, mean[c], marker='o', ms=3, label=c)
ax.axhline(0, color='black', lw=0.6)
ax.axvspan(pd.Timestamp('2026-06-14'), pd.Timestamp('2026-06-22'), color='grey', alpha=0.15,
           label='windows touching the recording gap')
ax.axvline(pd.Timestamp('2026-06-11'), color='grey', ls=':', lw=1, label='tournament starts')
ax.set_ylabel('mean HY correlation'); ax.set_xlabel('window start (3-day window)')
ax.legend(fontsize=8); plt.show()

## The full matrix for one window
Rows and columns are ordered by group, then match, so same-group pairs sit in blocks near the diagonal. Same-match pairs are blank (skipped).

In [ ]:
WIN = '2026-06-15'
ww = w[w.window_start == WIN]
keys = sorted(set(ww.a) | set(ww.b),
              key=lambda k: (series[tuple(k.split(':'))]['group'], k))
idx = {k: i for i, k in enumerate(keys)}
M = np.full((len(keys), len(keys)), np.nan)
for r in ww.itertuples():
    M[idx[r.a], idx[r.b]] = M[idx[r.b], idx[r.a]] = r.rho

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(M, cmap='RdBu_r', vmin=-0.4, vmax=0.4)
ax.set_xticks(range(len(keys))); ax.set_xticklabels(keys, rotation=90, fontsize=5)
ax.set_yticks(range(len(keys))); ax.set_yticklabels(keys, fontsize=5)
fig.colorbar(im, label='HY correlation'); ax.set_title(f'window starting {WIN}')
plt.show()

## Exercises — change one thing at a time
Do one, Restart & Run All, look, then undo it before the next. **Write each variant you try in `canonical/mine/variants.md`** (what you changed, what you saw). Later, when we test anything, the number of variants tried is what we have to correct for — an untracked variant is how researchers fool themselves.

1. **Mask the maintenance hours.** In Step 4–5 use `mask_maint=True` and re-run the walk (save under a different file name). Do the pre-tournament windows (June 2–10) change? Do the June 15–16 ones?
2. **Longer windows.** Before Step 7, set `hw.WINDOW = 5 * hw.DAY`. More data per window means less noise but less time detail. Does the June 15–16 pattern survive?
3. **Remove the spread filter.** Set `hw.MAX_SPREAD = 101` before Step 1 and re-plot the single match in Step 4–5 right at the final whistle. You should see the fake jump the filter exists to remove.
4. **Logit instead of arcsine.** Write a `logit(mid)` function (`log(p/(1-p))`, clip p to [0.005, 0.995]) and swap it in `build_series`. Logit amplifies 1¢ ticks near 0 and 100 — does anything blow up?
5. **See the Epps effect on real prices.** In a new cell, take two series from the June 15 window that HY says are correlated (pick a `same_team` pair from `ww`). Sample both mids on a 1-minute grid (last price at or before each minute), difference them, and compute `np.corrcoef`. Compare with the HY value. The grid number should be much closer to 0.